# 7주차 실습 — SQL (1) 조회 기초

**데이터베이스 관리 · 7주차 실습**

이 노트북은 수업 시간에 따라 하면서 채우는 연습용입니다. 점수는 없고, 과제 1(9주차 제출)의 바탕이 됩니다.

- 사용하는 데이터: `aact_mini` — ClinicalTrials.gov 임상시험 등록 자료(AACT)에서 2015~2024년 시작 시험 37,509건을 추린 수업용 데이터베이스
- 오늘 배우는 것: `SELECT` · `FROM` · `WHERE` · 비교 조건 · `AND/OR/NOT` · `IN` · `BETWEEN` · `LIKE` · `IS NULL` · `ORDER BY` · `LIMIT` · `DISTINCT`
- 사용법: 코드 셀을 위에서부터 차례로 실행합니다 (Shift+Enter). `%%sql` 로 시작하는 셀에는 SQL만 적습니다.

## 0. 준비
아래 셀을 실행하면 데이터가 연결됩니다. 오류가 나면 런타임을 다시 시작한 뒤 한 번 더 실행하세요.

In [ ]:
# ── 0. 준비: 이 셀은 그대로 실행만 하세요 (Shift+Enter) ─────────────────────────
import os, sys, urllib.request
DB  = "aact_mini.duckdb"
URL = "https://raw.githubusercontent.com/kevinkwshin/medical-db-course-2026/main/data/aact_mini.duckdb.gz"

# 로컬(수업 폴더)에서 열었을 때는 ../data/ 의 파일을 사용
for cand in [DB, os.path.join("..", "data", DB), os.path.join("data", DB)]:
    if os.path.exists(cand):
        DB = cand
        break
else:
    print("데이터 파일을 내려받습니다 (약 50MB, 1분 내외) ...")
    import gzip, shutil
    urllib.request.urlretrieve(URL, DB + ".gz")
    with gzip.open(DB + ".gz", "rb") as f_in, open(DB, "wb") as f_out:
        shutil.copyfileobj(f_in, f_out)
    os.remove(DB + ".gz")
    print("완료:", DB)

if "google.colab" in sys.modules:           # Colab 에서만 설치
    import subprocess; subprocess.run([sys.executable, "-m", "pip", "install", "-q", "duckdb", "jupysql", "duckdb-engine"])

import duckdb, pandas as pd, hashlib
pd.set_option("display.max_colwidth", 60)
%load_ext sql
%config SqlMagic.autopandas = True
%config SqlMagic.displaylimit = 30
%config SqlMagic.feedback = 0
conn = duckdb.connect(DB, read_only=True)
%sql conn --alias aact
print("연결 완료:", DB)

def _sig(df):
    """결과표의 내용 지문: 열 이름 소문자화, 소수 4자리 반올림, 행 정렬 후 해시"""
    d = df.copy()
    d.columns = [str(c).lower() for c in d.columns]
    for c in d.columns:
        if pd.api.types.is_float_dtype(d[c]):
            d[c] = d[c].round(4)
    d = d.astype(str).sort_values(by=list(d.columns)).reset_index(drop=True)
    return hashlib.md5(d.to_csv(index=False).encode()).hexdigest()[:12]

def check(name, df):
    """직전 %%sql 결과를 _ 로 넘겨 확인합니다.  예) check("Q1", _)"""
    if name not in ANSWERS:
        print("알 수 없는 문항:", name); return
    if not isinstance(df, pd.DataFrame):
        print("❌ 결과가 표가 아닙니다. %%sql 셀을 먼저 실행한 뒤, 바로 다음 셀에서 check('%s', _) 를 실행하세요." % name); return
    exp = ANSWERS[name]
    if _sig(df) == exp["sig"]:
        print(f"✅ {name} 정답!  (행 {len(df)}개)")
    else:
        hint = exp.get("hint", "")
        rows = exp.get("rows")
        msg = f"❌ {name} 아직 다릅니다."
        if rows is not None:
            msg += f"  기대 행 수 {rows}개, 현재 {len(df)}개."
        if hint:
            msg += "  힌트: " + hint
        print(msg)

In [ ]:
ANSWERS = {
 "Q1": {
  "sig": "6aa9b1794c4f",
  "hint": "overall_status = 'COMPLETED' AND phase = 'PHASE3'",
  "rows": 967
 },
 "Q2": {
  "sig": "a5e49089b208",
  "hint": "study_type = 'OBSERVATIONAL' AND enrollment >= 1000",
  "rows": 1
 },
 "Q3": {
  "sig": "25cac614ed6c",
  "hint": "BETWEEN '2022-01-01' AND '2022-12-31',  LIKE '%Nurs%'",
  "rows": 30
 },
 "Q4": {
  "sig": "aa3358328438",
  "hint": "IS NULL 을 사용",
  "rows": 1
 },
 "Q5": {
  "sig": "dde5fc01a6a9",
  "hint": "DISTINCT, ORDER BY",
  "rows": 2
 },
 "Q6": {
  "sig": "aae11c862925",
  "hint": "ORDER BY enrollment DESC LIMIT 5",
  "rows": 5
 },
 "Q7": {
  "sig": "e630f51b5596",
  "hint": "start_date >= '2024-01-01',  IN ('PHASE3', 'PHASE2/PHASE3')",
  "rows": 257
 },
 "Q8": {
  "sig": "1ee7cc0c6697",
  "hint": "minimum_age = '65 Years'  (문자이므로 따옴표)",
  "rows": 1
 }
}

## 1. 데이터 살펴보기 — 데이터 사전 먼저 (4주차)

분석을 시작할 때는 **어떤 테이블이 있고, 한 행이 무엇을 뜻하는지**부터 확인합니다.

In [ ]:
%%sql
-- 어떤 테이블이 있는지
SHOW TABLES

| 테이블 | 한 행이 뜻하는 것 |
|---|---|
| studies | 임상시험 1건 (기본키 `nct_id`) |
| conditions | 시험 1건의 대상 질환 1개 (여러 행) |
| eligibilities | 시험 1건의 선정·제외 기준 (1행) |
| facilities | 시험 1건이 진행되는 기관 1곳 (여러 행) |
| countries | 시험 1건이 진행되는 국가 1곳 (여러 행) |

오늘은 **studies** 한 테이블만 사용합니다.

In [ ]:
%%sql
-- studies 테이블의 열 이름과 데이터 타입 (데이터 사전)
DESCRIBE studies

In [ ]:
%%sql
-- 실제 값 5행만 보기  (* = 모든 열)
SELECT * FROM studies LIMIT 5

In [ ]:
%%sql
-- 행이 몇 개인지 = 임상시험이 몇 건인지
SELECT COUNT(*) AS 시험수 FROM studies

## 2. SELECT · FROM — 필요한 열만 고르기

```
SELECT 열이름1, 열이름2      -- 무엇을 볼지
FROM   테이블이름            -- 어디에서
```
`*` 대신 열 이름을 적으면 결과를 읽기 쉽습니다. `AS` 로 결과 열의 이름을 바꿀 수 있습니다.

In [ ]:
%%sql
-- 시험 번호, 제목, 시작일만
SELECT nct_id, brief_title, start_date
FROM studies
LIMIT 10

In [ ]:
%%sql
-- AS 로 열 이름 바꾸기
SELECT nct_id AS 등록번호, brief_title AS 제목, enrollment AS 등록인원
FROM studies
LIMIT 10

## 3. WHERE — 조건에 맞는 행만 남기기

```
SELECT ...
FROM   studies
WHERE  조건              -- 참(true)인 행만 남음
```

| 조건 종류 | 예 | 뜻 |
|---|---|---|
| 같다 / 다르다 | `overall_status = 'COMPLETED'`, `phase <> 'NA'` | 문자 값은 **작은따옴표** |
| 크기 비교 | `enrollment >= 1000` | 숫자는 따옴표 없이 |
| 날짜 비교 | `start_date >= '2020-01-01'` | 날짜는 `'YYYY-MM-DD'` 문자 형태 |
| 여러 조건 | `A AND B`, `A OR B`, `NOT A` | AND 가 OR 보다 먼저 계산됨 → 괄호로 묶기 |
| 목록 | `phase IN ('PHASE3', 'PHASE2/PHASE3')` | 여러 값 중 하나 |
| 범위 | `start_date BETWEEN '2020-01-01' AND '2020-12-31'` | 양 끝 포함 |
| 글자 패턴 | `brief_title LIKE '%Diabetes%'` | `%` = 아무 글자 여러 개, `_` = 한 글자 |
| 비어 있음 | `results_first_posted_date IS NULL` | `= NULL` 은 안 됨 |

In [ ]:
%%sql
-- 완료(COMPLETED)된 시험만
SELECT nct_id, brief_title, overall_status
FROM studies
WHERE overall_status = 'COMPLETED'
LIMIT 10

In [ ]:
%%sql
-- 등록 인원 1000명 이상
SELECT nct_id, brief_title, enrollment
FROM studies
WHERE enrollment >= 1000
LIMIT 10

In [ ]:
%%sql
-- 2024년 1월 1일 이후 시작  (날짜는 '작은따옴표' 문자 형태로)
SELECT nct_id, brief_title, start_date
FROM studies
WHERE start_date >= '2024-01-01'
LIMIT 10

In [ ]:
%%sql
-- 두 조건 모두: 3상이면서 완료
SELECT nct_id, brief_title, phase, overall_status
FROM studies
WHERE phase = 'PHASE3' AND overall_status = 'COMPLETED'
LIMIT 10

In [ ]:
%%sql
-- IN: 여러 값 중 하나  (3상 또는 2/3상)
SELECT nct_id, phase, start_date
FROM studies
WHERE phase IN ('PHASE3', 'PHASE2/PHASE3')
LIMIT 10

In [ ]:
%%sql
-- BETWEEN: 2020년 3월 한 달 동안 시작한 시험
SELECT nct_id, brief_title, start_date
FROM studies
WHERE start_date BETWEEN '2020-03-01' AND '2020-03-31'
LIMIT 10

In [ ]:
%%sql
-- LIKE: 제목에 Nurs 가 들어간 시험 (Nurse, Nursing 모두)
SELECT nct_id, brief_title
FROM studies
WHERE brief_title LIKE '%Nurs%'
LIMIT 10

### NULL — 값이 없다는 표시
`results_first_posted_date`(결과 게시일)가 비어 있으면 아직 결과가 게시되지 않은 시험입니다. 비어 있는지는 `IS NULL`, 채워져 있는지는 `IS NOT NULL` 로 확인합니다.

In [ ]:
%%sql
-- 완료됐지만 결과가 게시되지 않은 시험
SELECT nct_id, overall_status, completion_date, results_first_posted_date
FROM studies
WHERE overall_status = 'COMPLETED' AND results_first_posted_date IS NULL
LIMIT 10

In [ ]:
%%sql
-- 그런 시험이 몇 건인지
SELECT COUNT(*) AS 결과없음
FROM studies
WHERE overall_status = 'COMPLETED' AND results_first_posted_date IS NULL

## 4. ORDER BY · LIMIT · DISTINCT — 정렬, 개수 제한, 중복 제거

```
SELECT   ...
FROM     studies
WHERE    ...
ORDER BY 열이름 [ASC | DESC]   -- 오름차순(기본) / 내림차순
LIMIT    10                     -- 앞에서 10행만
```
**절(clause)의 순서는 정해져 있습니다**: `SELECT → FROM → WHERE → ORDER BY → LIMIT`

In [ ]:
%%sql
-- 등록 인원이 많은 순으로 10건
SELECT nct_id, brief_title, enrollment
FROM studies
ORDER BY enrollment DESC
LIMIT 10

> 위 결과의 1위 등록 인원을 보세요. 1억 8천만 명은 실제 등록 인원일 수 없습니다. 입력 오류(또는 인구 기반 관찰연구)로, **평균을 계산하면 이 한 건이 전체를 왜곡**합니다. 실제 데이터에는 이런 값이 늘 섞여 있습니다 (8·12주차에서 다룹니다).

In [ ]:
%%sql
-- 최근에 시작한 3상 시험 10건  (같은 날짜면 등록번호 순)
SELECT nct_id, brief_title, start_date
FROM studies
WHERE phase = 'PHASE3'
ORDER BY start_date DESC, nct_id
LIMIT 10

In [ ]:
%%sql
-- DISTINCT: 진행 상태(overall_status)에 어떤 값들이 있는지 중복 없이
SELECT DISTINCT overall_status
FROM studies
ORDER BY overall_status

In [ ]:
%%sql
-- 임상 단계(phase)의 값 종류  → NA 와 NULL(None)이 따로 있음에 주의
SELECT DISTINCT phase
FROM studies
ORDER BY phase

## 5. 자주 하는 실수 — 오류 메시지를 읽는 연습

아래 셀들은 **일부러 틀리게** 적었습니다. 실행해서 오류 메시지를 읽고, 바로 아래 셀에서 고친 버전을 실행해 비교하세요.
가장 위험한 실수는 **오류가 나지 않는 실수**입니다 (5-1, 5-6).

오류가 나면 붉은 글씨 중 `Original error message from DB driver:` 아래 줄이 실제 원인입니다. 그 줄만 읽으면 됩니다.

### 5-1. `= NULL` 은 오류가 나지 않지만 결과가 항상 0행

In [ ]:
%%sql
-- [오류 예시] 오류 없이 0 이 나옴 → 잘못된 결과를 믿게 됨
SELECT COUNT(*) AS n
FROM studies
WHERE results_first_posted_date = NULL

In [ ]:
%%sql
-- 고친 버전: IS NULL
SELECT COUNT(*) AS n
FROM studies
WHERE results_first_posted_date IS NULL

### 5-2. 문자 값에 작은따옴표를 빠뜨림

In [ ]:
%%sql
-- [오류 예시] COMPLETED 를 열 이름으로 해석 → Referenced column ... not found
SELECT nct_id
FROM studies
WHERE overall_status = COMPLETED
LIMIT 5

In [ ]:
%%sql
-- 고친 버전: 'COMPLETED'
SELECT nct_id
FROM studies
WHERE overall_status = 'COMPLETED'
LIMIT 5

### 5-3. 열 이름 오타

In [ ]:
%%sql
-- [오류 예시] brief_titel → 오류 메시지가 비슷한 열 이름을 제안해 줌 (Candidate bindings)
SELECT nct_id, brief_titel
FROM studies
LIMIT 5

In [ ]:
%%sql
-- 고친 버전
SELECT nct_id, brief_title
FROM studies
LIMIT 5

### 5-4. 절의 순서를 바꿈 (LIMIT 가 ORDER BY 앞에)

In [ ]:
%%sql
-- [오류 예시] syntax error → 순서: SELECT, FROM, WHERE, ORDER BY, LIMIT
SELECT nct_id, start_date
FROM studies
LIMIT 5
ORDER BY start_date

In [ ]:
%%sql
-- 고친 버전
SELECT nct_id, start_date
FROM studies
ORDER BY start_date
LIMIT 5

### 5-5. 날짜를 숫자처럼 비교

In [ ]:
%%sql
-- [오류 예시] 날짜(DATE)와 숫자(2020)는 비교할 수 없음
SELECT nct_id, start_date
FROM studies
WHERE start_date > 2020
LIMIT 5

In [ ]:
%%sql
-- 고친 버전: 'YYYY-MM-DD' 문자 형태
SELECT nct_id, start_date
FROM studies
WHERE start_date > '2020-12-31'
LIMIT 5

### 5-6. 대소문자 — LIKE 는 대소문자를 구분 (오류 없이 일부만 나옴)

In [ ]:
%%sql
-- [오류 예시] 소문자 covid 가 들어간 6건만 나옴 → 대문자 COVID 제목 2천여 건이 빠지지만 오류는 없음
SELECT COUNT(*) AS n
FROM studies
WHERE brief_title LIKE '%covid%'

In [ ]:
%%sql
-- 고친 버전: ILIKE 는 대소문자 무시 → 2,125건  (또는 UPPER(brief_title) LIKE '%COVID%')
SELECT COUNT(*) AS n
FROM studies
WHERE brief_title ILIKE '%covid%'

## 6. 직접 해보기

각 문항의 `%%sql` 셀에 SQL을 완성하고 실행한 뒤, **바로 다음 셀**의 `check(...)` 를 실행하면 정답 여부가 표시됩니다.
`___` 부분을 채우세요. 열 이름은 문항에 적힌 대로, 순서도 그대로 적습니다.

**Q1.** COMPLETED 상태이면서 PHASE3 인 시험의 `nct_id`, `brief_title` 을 조회하세요.

In [ ]:
%%sql
-- Q1
SELECT nct_id, brief_title
FROM studies
WHERE ___ AND ___

In [ ]:
check("Q1", _)

**Q2.** 등록 인원(`enrollment`)이 1000명 이상인 OBSERVATIONAL(관찰) 시험이 몇 건인지 `n` 이라는 이름으로 세세요.

In [ ]:
%%sql
-- Q2
SELECT COUNT(*) AS n
FROM studies
WHERE ___

In [ ]:
check("Q2", _)

**Q3.** 2022년에 시작한 시험 중 제목(`brief_title`)에 Nurs 가 들어간 시험의 `nct_id`, `brief_title`, `start_date` 를 조회하세요. (BETWEEN, LIKE)

In [ ]:
%%sql
-- Q3
SELECT nct_id, brief_title, start_date
FROM studies
WHERE start_date BETWEEN ___ AND ___
  AND brief_title LIKE ___

In [ ]:
check("Q3", _)

**Q4.** 완료(COMPLETED)되었지만 결과 게시일(`results_first_posted_date`)이 비어 있는 시험 수를 `n` 으로 세세요.

In [ ]:
%%sql
-- Q4
SELECT COUNT(*) AS n
FROM studies
WHERE ___

In [ ]:
check("Q4", _)

**Q5.** `study_type` 에 어떤 값이 있는지 중복 없이, 알파벳 순으로 조회하세요.

In [ ]:
%%sql
-- Q5
SELECT ___ study_type
FROM studies
___ study_type

In [ ]:
check("Q5", _)

**Q6.** 등록 인원이 가장 많은 시험 5건의 `nct_id`, `brief_title`, `enrollment` 를 많은 순으로 조회하세요.

In [ ]:
%%sql
-- Q6
SELECT nct_id, brief_title, enrollment
FROM studies
ORDER BY ___
LIMIT ___

In [ ]:
check("Q6", _)

**Q7.** 2024년에 시작한 시험 중 phase 가 PHASE3 또는 PHASE2/PHASE3 인 시험의 `nct_id`, `phase`, `start_date` 를 조회하세요. (IN)

In [ ]:
%%sql
-- Q7
SELECT nct_id, phase, start_date
FROM studies
WHERE start_date >= ___ AND phase IN (___)

In [ ]:
check("Q7", _)

**Q8.** `eligibilities` 테이블에서 최소 연령(`minimum_age`)이 '65 Years' 인 시험 수를 `n` 으로 세세요. (minimum_age 는 문자 열입니다)

In [ ]:
%%sql
-- Q8
SELECT COUNT(*) AS n
FROM eligibilities
WHERE ___

In [ ]:
check("Q8", _)

> Q8 뒤에 생각해 볼 것: `minimum_age` 가 문자이기 때문에 `'65 Years'` 처럼 정확히 같은 글자만 찾을 수 있습니다. "65세 이상"을 찾으려면 숫자 열(`calculated_values.minimum_age_num`)이 필요합니다 — 5주차 데이터 타입, 12주차 오류에서 다시 봅니다.

## 7. 오늘 배운 것

| 절 | 역할 | 기억할 점 |
|---|---|---|
| `SELECT` | 어떤 열을 볼지 | `*` 보다 열 이름, `AS` 로 이름 바꾸기 |
| `FROM` | 어느 테이블에서 | 먼저 한 행의 뜻을 확인 |
| `WHERE` | 어떤 행만 남길지 | 문자·날짜는 `'따옴표'`, 비어 있음은 `IS NULL` |
| `ORDER BY` | 정렬 | `DESC` 내림차순 |
| `LIMIT` | 앞에서 n행 | 항상 마지막 |
| `DISTINCT` | 중복 제거 | 값의 종류를 볼 때 |

**절의 순서**: `SELECT → FROM → WHERE → ORDER BY → LIMIT`

다음 시간(8주차): 집계(COUNT · AVG · MIN · MAX), GROUP BY, 그리고 두 테이블을 연결하는 JOIN.